# Session 1 · What is an LLM: next-token prediction

A Large Language Model does **one thing only**: you give it some text and it returns **a probability distribution over the next token**.

| Input | Possible next tokens |
|---|---|
| `The capital of Greece is` | `Athens` 0.93 · `the` 0.03 · `a` 0.01 · … |

Mathematically: $P(\text{token}_{t+1} \mid \text{token}_1, \dots, \text{token}_t)$

**So how does it write whole answers?** By repetition (autoregressive generation):
1. it predicts the next token,
2. appends it to the end of the text,
3. asks the model again with the longer text, over and over.

**How is it trained?** We give it lots of text and "penalise" it whenever it assigns a low probability to the token that *actually* comes next. No labels are needed: **the labels are the text itself, shifted by one position** (self-supervised learning).

In this session we see this in practice with the simplest "language model" there is: **counting**.

## Step 1 · Setup
We load the libraries and choose where the computations will run:
- `torch` is PyTorch. We don't use it yet in this session, but it is the foundation from Session 2 onwards.
- `torch.manual_seed(42)` makes "random" results reproducible: every run gives the same output.
- `device` becomes `"cuda"` if a GPU is available (in Colab: Runtime → Change runtime type → T4 GPU), otherwise `"cpu"`.

In [ ]:
import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

## Step 2 · The dataset: Tiny Shakespeare
We download about **1 million characters** from Shakespeare's plays as a plain text file.
- It is small enough to train quickly and "recognisable" enough to tell whether the model is learning.
- The `if not os.path.exists(...)` check downloads it only the first time.
- At the end, the whole dataset is **one long string** in the variable `text`.

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(URL, "input.txt")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print("Χαρακτήρες στο dataset:", len(text))

## Step 3 · A first look at the data
Always look at the data before doing anything else. Notice the format: a speaker's name followed by a colon, then their lines, with blank lines in between. This is the format we want our model to learn to imitate.

In [ ]:
print(text[:400])

## Step 4 · Our first "language model": counting which character follows which
For every pair of consecutive characters `(a, b)` in the text, we increase the counter "`b` came after `a`" by 1.
- `zip(text, text[1:])` gives all the pairs: the text and the same text **shifted by 1**. This is exactly the idea behind training: input = the text, target = what comes next.
- `next_counts[a]` is a `Counter`: for the character `a`, how many times each other character followed it.

This is already a (very simple) language model: it knows $P(\text{next} \mid \text{current})$. After `q`, the next character is almost always `u`.

In [ ]:
from collections import Counter, defaultdict

next_counts = defaultdict(Counter)
for a, b in zip(text, text[1:]):
    next_counts[a][b] += 1

print("Μετά το 'q':", next_counts["q"].most_common(3))
print("Μετά το 't':", next_counts["t"].most_common(5))

## Step 5 · Generation with greedy decoding
We make the model **write**: we start from `"T"` and, 40 times,
1. look at the **last** character (`s[-1]`),
2. take the **most likely** next one (`most_common(1)`),
3. append it to the end and start again.

This is called **greedy** decoding. We will see it get stuck in a loop (`the the the…`): the same character always produces the same next one.

In [ ]:
def predict_next(ch):
    return next_counts[ch].most_common(1)[0][0]

s = "T"
for _ in range(40):
    s += predict_next(s[-1])
print(s)

## Step 6 · Generation with sampling
Instead of the most likely character, we **pick at random, weighted by the probabilities**: a character that followed 100 times is 10 times more likely to be chosen than one that followed 10 times.
- `random.choices(..., weights=...)` does exactly this weighted random choice.

The text now has variety and a shape that looks like Shakespeare (names, colons, line breaks), but the words are still gibberish.

### What we learned
- **Generation = predict the next token + repeat.** This is what ChatGPT and Claude do too.
- **Greedy vs sampling:** greedy is predictable but gets stuck; sampling has variety but also "madness". In Session 4 we will see the tools that balance the two (`temperature`, `top_k`, `top_p`).
- **The big problem:** the model only sees **the last character**. It has no idea what was said before.

> **The idea of the whole course:** we will replace this "table of counts" with a **neural network that sees all the previous text**. That network is the **Transformer**.

In [ ]:
def sample_next(ch):
    options = next_counts[ch]
    return random.choices(list(options.keys()), weights=list(options.values()))[0]

s = "T"
for _ in range(200):
    s += sample_next(s[-1])
print(s)